# Figure calibration helper (Pass 3)

Use this notebook to turn an **auto-traced** figure (pixel space) into a
**calibrated** data-space CSV. Workflow:
1. pick a figure from a catalogue (`catalogue_id`, `pdf_page_number`);
2. auto-trace to find the plot rectangle;
3. read four tick references off the axes;
4. build a `CalibrationSpec`, re-digitise, save CSV + overlay;
5. eyeball the overlay and the recovered ranges.

See `docs/RUNBOOK.md` §5 for the full explanation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from ntrs_extractor import figures, catalogue_io
import matplotlib.pyplot as plt

# --- choose the target figure -------------------------------------------
PDF  = '../data/raw/apollo/Medical_Results_of_Apollo_14.pdf'
PAGE = 4                      # pdf_page_number from the catalogue row
CAT_ID = 'A14-F-001'
OUT_CSV = f'../data/outputs/apollo/figures/{CAT_ID}.csv'
OVERLAY = f'../qc/apollo/overlays/{CAT_ID}.png'

In [ ]:
# --- 1) auto-trace to find the plot rectangle (pixel space) -------------
r = figures.digitise_figure(PDF, PAGE)
x0, y0, x1, y1 = r.plot_bbox_px
print('plot rect px (left,top,right,bottom):', r.plot_bbox_px)
print('series traced:', {k: len(v) for k, v in r.series.items()})
plt.figure(figsize=(7, 9)); plt.imshow(r.overlay); plt.axis('off');
plt.title('auto-trace overlay (check dots sit on the curves)'); plt.show()

In [ ]:
# --- 2) calibrate: read TWO x ticks and TWO y ticks off the axes --------
#   image y grows downward, so bottom tick (y1) is the SMALLER data value.
cal = figures.CalibrationSpec(
    x_px=(x0, x1), x_val=(1, 10),      # e.g. minutes: 1 at left, 10 at right
    y_px=(y1, y0), y_val=(50, 120),    # e.g. bpm: 50 at bottom, 120 at top
    x_log=False, y_log=False,
)
r2 = figures.digitise_figure(PDF, PAGE, calibration=cal)
figures.save_figure(r2, OUT_CSV, overlay_png=OVERLAY)
print('calibrated:', r2.calibrated, '| points:', r2.n_points)
for name, arr in r2.series.items():
    print(f"{name}: x[{arr[:,0].min():.2f}..{arr[:,0].max():.2f}] "
          f"y[{arr[:,1].min():.1f}..{arr[:,1].max():.1f}]")

In [ ]:
# --- 3) plot the recovered series in data units -------------------------
import pandas as pd
df = pd.read_csv(OUT_CSV)
for name, g in df.groupby('series'):
    plt.plot(g['x'], g['y'], '.', ms=2, label=name)
plt.xlabel('x (data units)'); plt.ylabel('y (data units)'); plt.legend(); plt.show()
df.head()